### CRISP-DM Phase 5.2 - Evaluation : Statistical significance

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import spearmanr

In [2]:
# Load the datasets
correlation_country = pd.read_csv('outputs/correlation_country.csv')
correlation_continent = pd.read_csv('outputs/correlation_continent.csv', keep_default_na=False)
# North America's code is NA and was considered as NaN
legislative_coverage = pd.read_csv('../law/outputs/legislative_coverage_country.csv')
hazard_intensity = pd.read_csv('../sensor/outputs/hazard_intensity_country.csv')

Statistical significance

In [ ]:
def statistical_significance(row):
    if row['p_value'] >= 0.05:
        return 'Not significant'
    elif row['Rho'] > 0.3:
        return 'Strong positive relationship'
    elif row['Rho'] < -0.3:
        return 'Strong negative relationship'
    else:
        return 'Weak relationship'

correlation_country['Result'] = correlation_country.apply(statistical_significance, axis=1)
print(f"Country-level correlation results:\n{correlation_country['Result'].value_counts()}")

correlation_continent['Result'] = correlation_continent.apply(statistical_significance, axis=1)
print(f"Continent-level correlation results:\n{correlation_continent['Result'].value_counts()}")

In [ ]:
# By hazard
print(f"Country-level result classification by hazard:\n{correlation_country.groupby(['Hazard', 'Result'])['Country'].count().unstack(fill_value=0)}")
print(f"Continental-level result classification by hazard:\n{correlation_continent.groupby(['Hazard', 'Result'])['Continent'].count().unstack(fill_value=0)}")

Correlation analysis

In [6]:
## Distribution of rho
# Country
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(correlation_country['Rho'], bins=30, edgecolor='white')
axes[0].set_xlabel('Rho')
axes[0].set_ylabel('Count')

significant = correlation_country[correlation_country['p_value'] < 0.05]['Rho']
non_significant = correlation_country[correlation_country['p_value'] >= 0.05]['Rho']

bins_country = np.linspace(correlation_country['Rho'].min(), correlation_country['Rho'].max(), 31)
axes[1].hist(non_significant, bins=bins_country, alpha=0.6, color='grey', edgecolor='white', label='Not significant')
axes[1].hist(significant, bins=bins_country, alpha=0.6, edgecolor='white', label='Significant')
axes[1].set_xlabel('Rho')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/5_rho_distribution_country.png', dpi=150, bbox_inches='tight')
plt.close()

# Continent
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(correlation_continent['Rho'], bins=15, edgecolor='white')
axes[0].set_xlabel('Rho')
axes[0].set_ylabel('Count')

significant = correlation_continent[correlation_continent['p_value'] < 0.05]['Rho']
non_significant = correlation_continent[correlation_continent['p_value'] >= 0.05]['Rho']

bins_continent = np.linspace(correlation_continent['Rho'].min(), correlation_continent['Rho'].max(), 16)
axes[1].hist(non_significant, bins=bins_continent, alpha=0.6, color='grey', edgecolor='white', label='Not significant')
axes[1].hist(significant, bins=bins_continent, alpha=0.6, edgecolor='white', label='Significant')
axes[1].set_xlabel('Rho')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/5_rho_distribution_continent.png', dpi=150, bbox_inches='tight')
plt.close()

In [ ]:
## Mean rho per hazard 
# Country
fig, ax = plt.subplots(figsize=(5, 4))
hazard_mean_rho_country = correlation_country.groupby('Hazard')['Rho'].mean().sort_values()
colors_country = ['red' if r < 0 else 'green' for r in hazard_mean_rho_country.values]
ax.barh(hazard_mean_rho_country.index, hazard_mean_rho_country.values, color=colors_country, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Mean rho')
plt.tight_layout()
plt.savefig('outputs/5_mean_rho_country.png', dpi=150, bbox_inches='tight')
plt.close()

# Continent
fig, ax = plt.subplots(figsize=(5, 4))
hazard_mean_rho_continent = correlation_continent.groupby('Hazard')['Rho'].mean().sort_values()
colors_continent = ['red' if r < 0 else 'green' for r in hazard_mean_rho_continent.values]
ax.barh(hazard_mean_rho_continent.index, hazard_mean_rho_continent.values, color=colors_continent, edgecolor="white")
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Mean rho')
plt.tight_layout()
plt.savefig("outputs/5_mean_rho_continent.png", dpi=150, bbox_inches="tight")
plt.close()

In [ ]:
# Correlation comparison
country_summary = correlation_country.groupby('Hazard')['Rho'].mean().rename('country_mean_rho')
continent_summary = correlation_continent.groupby('Hazard')['Rho'].mean().rename('continent_mean_rho')
comparison = pd.concat([country_summary, continent_summary], axis=1)
comparison['difference'] = comparison['continent_mean_rho'] - comparison['country_mean_rho']
print(comparison.sort_values('difference', ascending=False))

Case study

In [ ]:
europe = ['ALB', 'AND', 'AUT', 'BEL', 'BGR', 'BIH', 'BLR', 'CHE', 'CYP', 'CZE', 'DEU', 'DNK', 'ESP', 
          'EST', 'FIN', 'FRA', 'GBR', 'GRC', 'HRV', 'HUN', 'IRL', 'ITA', 'LIE', 'LTU', 'LUX', 'LVA', 
          'MDA', 'MKD', 'MLT', 'MNE', 'NLD', 'NOR', 'POL', 'PRT', 'ROU', 'SMR', 'SRB', 'SVK', 'SVN', 
          'SWE', 'UKR', 'VAT']
# excluding Russia and Turkiye (between two contients) and Iceland (too far)

australia = ['AUS']
case_study_countries = europe + australia

In [ ]:
hazard_variable_dict = {'flood': 'Total_precipitation', 'drought': 'SPEI', 
                        'temperature_extremes': '2m_temperature', 'sea_level_rise': 'Sea_level_anomaly', 
                        'storm': 'Instantaneous_wind_gust', 'melting': 'Snowmelt'}

In [ ]:
# Correlation comparison
case_study_corr = []

for hazard, variable in hazard_variable_dict.items():
    for country in case_study_countries:
        country_intensity = hazard_intensity[hazard_intensity['Country'] == country]
        country_coverage = legislative_coverage[(legislative_coverage['Country'] == country) & (legislative_coverage['Hazard'] == hazard)][['Year', 'Count', 'Count_national']].copy()
        
        data = pd.merge(country_intensity[['Year', variable]], country_coverage, on='Year', how='inner').dropna()

        n_members = 27
        data['Count_weighted'] = data['Count_national'] + (data['Count'] - data['Count_national']) / n_members

        if data[variable].nunique() == 1 or data['Count_weighted'].nunique() == 1:
            continue
        try:
            rho, p = spearmanr(data[variable], data['Count_weighted'])
            case_study_corr.append({
                'Hazard': hazard, 'Country': country, 
                'Rho': rho, 'p_value': p,
                'Group': 'Europe' if country in europe else 'Australia'
            })
        except Exception as e:
            continue

case_study_df = pd.DataFrame(case_study_corr)
aus_results = case_study_df[case_study_df['Group'] == 'Australia'].set_index('Hazard')[['Rho', 'p_value']]
eur_results = case_study_df[case_study_df['Group'] == 'Europe'].groupby('Hazard')[['Rho', 'p_value']].mean()
comparison = aus_results.join(eur_results, lsuffix='_AUS', rsuffix='_EUR')
print(comparison)

In [ ]:
# Count significant countries in Europe
eu_df = case_study_df[case_study_df['Group'] == 'Europe']

for hazard in hazard_variable_dict.keys():
    eu_sig = eu_df[(eu_df['Hazard'] == hazard) & (eu_df['p_value'] < 0.05)]
    print(f"{hazard}: {len(eu_sig)}")